In [28]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json 
from statistics import mean

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [29]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [94]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [95]:
dataset=generate_dataset()
print(dataset)

[{'task': 'Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression', 'format': 'regex', 'solution_criteria': 'The regex must correctly capture three groups: ISO 8601 timestamp, log level (DEBUG, INFO, WARN, ERROR), and the remaining message text. Should handle multiline log formats.'}, {'task': 'Create a JSON configuration object for an AWS Lambda function that specifies environment variables, timeout settings, memory allocation, and VPC configuration', 'format': 'json', 'solution_criteria': 'The JSON must include valid keys for environment variables (as key-value pairs), Timeout (in seconds), MemorySize (128-10240 MB), and VpcConfig with SubnetIds and SecurityGroupIds arrays. Must be properly formatted and valid JSON.'}, {'task': 'Write a Python function that takes an AWS S3 object key path and returns the bucket name, folder structure, and file name as separate components', 'format': 'python', 'solution_criteria': "Function must 

In [96]:
# save dataset to file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [81]:
# Code base grading
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except:
        print("regex error")
        return 0
    
def grade_syntax(test_case, output):
    format = test_case["format"]
    if format == "json":
        return validate_json(output)
    if format == "python":
        return validate_python(output)
    else:
        return validate_regex(output)

In [91]:
# Helper functions for evaluating prompts

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt=f"""
        Please solve the following task:

        {test_case["task"]}
        * Respond only with python, JSON or a plain Regex
        * Do not add any comments or comentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    Be sure to follow the criterias according to Solution Criteria
    
    Task: {test_case["task"]}
    Solution Criteria: {test_case["solution_criteria"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    syntax_score = grade_syntax(test_case, output)
    score = mean([model_score, syntax_score])

    return {
        "test_case": test_case,
        "output": output,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

def run_eval(dataset):
    """Loads the datase and calls run_test_case with each case"""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [92]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.166666666666666


In [93]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression",
      "format": "regex",
      "solution_criteria": "The regex must correctly capture three groups: (1) ISO 8601 timestamp, (2) log level (INFO, ERROR, WARNING, DEBUG), and (3) the message content. It should handle variations in spacing and work with standard CloudWatch log formats."
    },
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'^(\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}:\\d{2}\\.\\d{3}Z)\\s+\\[([A-Z]+)\\]\\s+(.+)$'\n    match = re.search(pattern, log_entry)\n    \n    if match:\n        return {\n            'timestamp': match.group(1),\n            'log_level': match.group(2),\n            'message': match.group(3)\n        }\n    return None\n\nlog_entry = \"2024-01-15T10:30:45.123Z [INFO] Application started successfully\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dump